In [20]:
# Ingest current data stream
historical_df = df.copy()
historical_df['Date'] = pd.to_datetime(historical_df['date'])
historical_df['Year'] = historical_df['Date'].dt.year

# Rename columns to match expected format
rename_dict = {
    'revenue': 'Revenue',
    'costOfRevenue': 'Cost of Revenue',
    'grossProfit': 'Gross Profit',
    'operatingIncome': 'Operating Income',
    'netIncome': 'Net Income',
    'ebitda': 'EBITDA',
    'interestExpense': 'Interest Expense',
    'eps': 'EPS',
    'weightedAverageShsOut': 'Shares Outstanding'
}

historical_df = historical_df.rename(columns=rename_dict)

# Drop the lowercase 'date' column to avoid duplication
if 'date' in historical_df.columns:
    historical_df = historical_df.drop(columns=['date'])

# Convert historical financial values to numeric
for col in ['Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Income', 'Net Income']:
    if col in historical_df.columns:
        historical_df[col] = pd.to_numeric(historical_df[col], errors='coerce')

# Load the richer financial history only for ratio-based forecasting
fs_df = pd.read_csv('FS_data.csv')
fs_df['date'] = pd.to_datetime(fs_df['date'])
fs_df = fs_df.sort_values('date')
fs_df['Year'] = fs_df['date'].dt.year

fs_df = fs_df[['Year', 'revenue', 'costOfRevenue', 'grossProfit', 'operatingIncome', 'netIncome', 'ebitda', 'interestExpense', 'eps', 'weightedAverageShsOut']]
fs_df = fs_df.rename(columns=rename_dict)

for col in ['Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'EBITDA', 'Interest Expense', 'EPS', 'Shares Outstanding']:
    if col in fs_df.columns:
        fs_df[col] = pd.to_numeric(fs_df[col], errors='coerce')

# Use only the Apple history for the regression and the main financial values
regression_df = historical_df.dropna(subset=['Revenue', 'Cost of Revenue']).copy()
regression_df = regression_df[['Year', 'Revenue', 'Cost of Revenue', 'Operating Income', 'Net Income']]

# Fit separate regressions for revenue and cost patterns
X = regression_df[['Year']].values
y_rev = regression_df['Revenue'].values
y_cost = regression_df['Cost of Revenue'].values

model_rev = LinearRegression().fit(X, y_rev)
model_cost = LinearRegression().fit(X, y_cost)

# Build a 5-year forward horizon framework
future_years = np.array([[regression_df['Year'].max() + i] for i in range(1, 6)])
pred_rev = model_rev.predict(future_years)
pred_cost = model_cost.predict(future_years)

# Historical margins from the Apple data
operating_margin = regression_df['Operating Income'].div(regression_df['Revenue']).mean()
net_margin = regression_df['Net Income'].div(regression_df['Revenue']).mean()

# Forecast ratios from the data (used only for the extra metrics)
interest_expense_margin = fs_df['Interest Expense'].div(fs_df['Revenue']).mean() if fs_df['Interest Expense'].notna().any() and fs_df['Revenue'].notna().any() else 0.01
ebitda_margin = fs_df['EBITDA'].div(fs_df['Revenue']).mean() if fs_df['EBITDA'].notna().any() and fs_df['Revenue'].notna().any() else operating_margin
shares_outstanding = fs_df['Shares Outstanding'].mean() if fs_df['Shares Outstanding'].notna().any() else 1

# Build forecast dataframe with all matching columns from historical_df
forecast_dict = {
    'Date': [pd.to_datetime(f"{int(y[0])}-09-30") for y in future_years],
    'Year': future_years.flatten(),
    'symbol': 'AAPL',
    'reportedCurrency': 'USD',
    'Revenue': pred_rev,
    'Cost of Revenue': pred_cost,
    'Gross Profit': pred_rev - pred_cost,
    'Operating Income': pred_rev * operating_margin,
    'Net Income': pred_rev * net_margin,
    'Interest Expense': pred_rev * interest_expense_margin,
    'EBITDA': pred_rev * ebitda_margin,
    'EPS': (pred_rev * net_margin) / shares_outstanding,
    'Shares Outstanding': shares_outstanding,
}

forecast_df = pd.DataFrame(forecast_dict)

# Add IsForecast column to both dataframes
historical_df['IsForecast'] = 0
forecast_df['IsForecast'] = 1

# Get all columns from historical_df to ensure forecast_df has all of them
for col in historical_df.columns:
    if col not in forecast_df.columns:
        forecast_df[col] = None

# Exclude empty forecast-only fields before concatenation
forecast_df = forecast_df.dropna(axis=1, how='all')

# Concatenate historical and forecast data, keeping column order from historical_df
result_dataframe = pd.concat([historical_df, forecast_df], ignore_index=True, sort=False)

# Remove completely empty columns (columns that are all NaN)
result_dataframe = result_dataframe.dropna(axis=1, how='all')

# Reorder columns: keep Date, Year, IsForecast at the end, rest alphabetically
core_cols = ['Date', 'Year', 'IsForecast']
other_cols = sorted([c for c in result_dataframe.columns if c not in core_cols])
result_dataframe = result_dataframe[other_cols + core_cols]
result_dataframe['period'] = result_dataframe['period'] = 'FY'
result_dataframe['cik'] = result_dataframe['cik'] = 320193
result_dataframe.head()

,Cost of Revenue,EBITDA,EPS,Gross Profit,Interest Expense,Net Income,Operating Income,Revenue,Shares Outstanding,acceptedDate,...,reportedCurrency,researchAndDevelopmentExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,symbol,totalOtherIncomeExpensesNet,weightedAverageShsOutDil,Date,Year,IsForecast
0,2.209600e+11,1.444270e+11,7.49,1.952010e+11,0.000000e+00,1.120100e+11,1.330500e+11,4.161610e+11,1.494850e+10,10/31/2025 6:01:26 AM,...,USD,3.455000e+10,0.000000e+00,2.760100e+10,AAPL,-321000000.0,1.500470e+10,2025-09-27,2025,0
1,2.103520e+11,1.349300e+11,6.11,1.806830e+11,0.000000e+00,9.373600e+10,1.232160e+11,3.910350e+11,1.534378e+10,11/1/2024 6:01:36 AM,...,USD,3.137000e+10,1.863900e+10,2.609700e+10,AAPL,269000000.0,1.540810e+10,2024-09-28,2024,0
2,2.141370e+11,1.291880e+11,6.16,1.691480e+11,3.933000e+09,9.699500e+10,1.143010e+11,3.832850e+11,1.574423e+10,11/2/2023 6:08:27 PM,...,USD,2.991500e+10,0.000000e+00,2.493200e+10,AAPL,-565000000.0,1.581255e+10,2023-09-30,2023,0
3,2.235460e+11,1.331380e+11,6.15,1.707820e+11,2.931000e+09,9.980300e+10,1.194370e+11,3.943280e+11,1.621596e+10,10/27/2022 6:01:14 PM,...,USD,2.625100e+10,0.000000e+00,2.509400e+10,AAPL,-334000000.0,1.632582e+10,2022-09-24,2022,0
4,2.129810e+11,1.231360e+11,5.67,1.528360e+11,2.645000e+09,9.468000e+10,1.089490e+11,3.658170e+11,1.670127e+10,10/28/2021 6:04:28 PM,...,USD,2.191400e+10,0.000000e+00,2.197300e+10,AAPL,258000000.0,1.686492e+10,2021-09-25,2021,0


In [21]:
result_dataframe.to_csv('result_dataframe.csv', index=False)